# UCS420: Cognitive Computing
## Assignment 4 – A Cognitive FAQ System Using Pandas
**Roll Number:** 24

### Environment Setup

In [ ]:
import pandas as pd
import os

### Q1: Build Your Personalized Knowledge Base

**Rules applied for Roll Number 24:**
- Last two digits are `2` and `4`.
- **Digit 2**: category `["billing", "account", "general"][2 % 3]` -> `general`. Question: "what is your customer support contact information", Answer: "You can email support at contact@college.edu or call +1-800-555-0199.", Keywords: "contact support helpline".
- **Digit 4**: category `["billing", "account", "general"][4 % 3]` -> `account`. Question: "how do i update my registered mobile number", Answer: "You can update your registered mobile number under Account Settings > Profile Info.", Keywords: "update register mobile number phone".

In [ ]:
fixed_entries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing"
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account"
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general"
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing"
    }
]

personalized_entries = [
    # Digit 2: Category "general"
    {
        "question": "what is your customer support contact information",
        "answer": "You can email support at contact@college.edu or call +1-800-555-0199.",
        "keywords": "contact support helpline",
        "category": "general"
    },
    # Digit 4: Category "account"
    {
        "question": "how do i update my registered mobile number",
        "answer": "You can update your registered mobile number under Account Settings > Profile Info.",
        "keywords": "update register mobile number phone",
        "category": "account"
    }
]

# Combine into a single DataFrame
df = pd.DataFrame(fixed_entries + personalized_entries)
print("Personalized FAQ DataFrame:")
df

### Q2: Generate and Score a Hypothesis
Implement a scoring function that takes a query string and returns all matching entries ranked by confidence.

In [ ]:
def score_query(query, df):
    query_words = set(query.lower().strip().split())
    results = []
    for idx, row in df.iterrows():
        keyword_words = set(row['keywords'].lower().split())
        score = len(query_words.intersection(keyword_words))
        if score > 0:
            results.append({
                "index": idx,
                "question": row['question'],
                "answer": row['answer'],
                "category": row['category'],
                "score": score
            })
    res_df = pd.DataFrame(results)
    if not res_df.empty:
        res_df = res_df.sort_values(by='score', ascending=False).reset_index(drop=True)
    return res_df

print("Matching entries for query 'how to pay':")
score_query("how to pay", df)

### Q3: Get Questions in Same Category
Write a function `same_category(category_name, df)` that returns all questions belonging to a given category. Call it using the category of one of the personalized entries from Q1.

In [ ]:
def same_category(category_name, df):
    filtered_df = df[df['category'].str.lower() == category_name.lower()]
    return filtered_df['question']

test_category = df.loc[4, 'category'] # index 4 is category "general"
print(f"Questions in category '{test_category}':")
same_category(test_category, df)

### Q4: Ask for a New Keyword and Save to CSV
Pick any one entry, append a user-provided keyword, and save the DataFrame to a CSV file named `<your_roll_number>_faq_data.csv`.

In [ ]:
def add_keyword_and_save(df, entry_index, new_keyword, roll_number="24"):
    current_keywords = df.loc[entry_index, 'keywords']
    keywords_list = current_keywords.split()
    if new_keyword.strip().lower() not in [k.lower() for k in keywords_list]:
        df.loc[entry_index, 'keywords'] = f"{current_keywords} {new_keyword.strip()}"
    
    filename = f"{roll_number}_faq_data.csv"
    df.to_csv(filename, index=False)
    print(f"Keyword '{new_keyword}' added to entry {entry_index}.")
    print(f"DataFrame saved to '{filename}' successfully!")

# Ask user for a keyword (we will prompt or run with a mock value for script compatibility)
new_keyword = input("Enter a new keyword: ") if 'IPython' in globals() else "billing_support"
add_keyword_and_save(df, 5, new_keyword)

### Q5: FAQ Count per Category (Groupby)
Using groupby, print how many FAQ entries you have per category.

In [ ]:
print("FAQ Entries count per category:")
df.groupby('category').size()

### Q6: Handle Ties in Scoring Function
Modify the scoring function so that if two or more entries tie for the highest score, it prints all matching entries instead of picking one. Show a demonstration query with a tie and one without.

In [ ]:
def score_query_v2(query, df):
    query_words = set(query.lower().strip().split())
    results = []
    for idx, row in df.iterrows():
        keyword_words = set(row['keywords'].lower().split())
        score = len(query_words.intersection(keyword_words))
        if score > 0:
            results.append({
                "index": idx,
                "question": row['question'],
                "answer": row['answer'],
                "category": row['category'],
                "score": score
            })
    res_df = pd.DataFrame(results)
    if res_df.empty:
        print(f"Query: '{query}' -> No matches found.")
        return res_df
        
    res_df = res_df.sort_values(by='score', ascending=False).reset_index(drop=True)
    max_score = res_df['score'].max()
    highest_matches = res_df[res_df['score'] == max_score]
    
    if len(highest_matches) > 1:
        print(f"Query: '{query}' -> [TIE DETECTED] Multiple entries matched with highest score {max_score}:")
        for _, row in highest_matches.iterrows():
            print(f"  - Question: '{row['question']}' (Category: {row['category']})")
    else:
        print(f"Query: '{query}' -> [SINGLE BEST MATCH] Found with score {max_score}:")
        row = highest_matches.iloc[0]
        print(f"  - Question: '{row['question']}' (Category: {row['category']})")
    return res_df

print("--- Test 1: Query causing a tie (matches 'fee' keyword in multiple entries) ---")
score_query_v2("fee", df)

print("\n--- Test 2: Query with single match (no tie) ---")
score_query_v2("password login", df);